In [1]:
import pandas as pd
import numpy as np

TRAIN_FILE = "../data/processed/train_data.csv"
TEST_FILE = "../data/processed/test_data.csv"

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (6082007, 25)
Test shape: (89466, 25)


In [2]:
train_df = pd.get_dummies(
    train_df,
    columns=["type"],
    dtype=int
)

test_df = pd.get_dummies(
    test_df,
    columns=["type"],
    dtype=int
)

test_df = test_df.reindex(
    columns=train_df.columns,
    fill_value=0
)

In [3]:
ANOMALY_FEATURES = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud",

    "user_transaction_count_before",
    "previous_transaction_amount",
    "time_since_previous_transaction",
    "previous_average_amount",
    "amount_deviation",
    "amount_to_previous_average",

    "balance_depletion",
    "amount_to_balance_ratio",

    "receiver_transaction_count_before",
    "receiver_previous_amount",
    "receiver_transaction_frequency",

    "user_transfer_count_before",
    "user_cashout_count_before",

    "transaction_velocity",

    "type_CASH_IN",
    "type_CASH_OUT",
    "type_DEBIT",
    "type_PAYMENT",
    "type_TRANSFER"
]

print("Number of anomaly features:", len(ANOMALY_FEATURES))

Number of anomaly features: 26


In [4]:
X_train_anomaly = train_df[ANOMALY_FEATURES].fillna(0)
X_test_anomaly = test_df[ANOMALY_FEATURES].fillna(0)

y_test_anomaly = test_df["isFraud"]

print("Training anomaly data:", X_train_anomaly.shape)
print("Test anomaly data:", X_test_anomaly.shape)

print(
    "Remaining NaN:",
    X_train_anomaly.isna().sum().sum()
)

Training anomaly data: (6082007, 26)
Test anomaly data: (89466, 26)
Remaining NaN: 0


In [5]:
from sklearn.ensemble import IsolationForest

isolation_model = IsolationForest(
    n_estimators=200,
    contamination=0.001,
    random_state=42,
    n_jobs=-1
)

print("Isolation Forest created.")

Isolation Forest created.


In [6]:
isolation_model.fit(X_train_anomaly)

print("Isolation Forest training completed.")

Isolation Forest training completed.


In [7]:
test_anomaly_score = isolation_model.decision_function(
    X_test_anomaly
)

print("Minimum anomaly score:", test_anomaly_score.min())
print("Maximum anomaly score:", test_anomaly_score.max())

Minimum anomaly score: -0.051293284851127385
Maximum anomaly score: 0.2881922204242946


In [9]:
test_anomaly_risk = -test_anomaly_score

print("Minimum anomaly risk:", test_anomaly_risk.min())
print("Maximum anomaly risk:", test_anomaly_risk.max())

Minimum anomaly risk: -0.2881922204242946
Maximum anomaly risk: 0.051293284851127385


In [10]:
print(
    "Average anomaly risk - Legitimate:",
    test_anomaly_risk[y_test_anomaly == 0].mean()
)

print(
    "Average anomaly risk - Fraud:",
    test_anomaly_risk[y_test_anomaly == 1].mean()
)

Average anomaly risk - Legitimate: -0.23997336734264765
Average anomaly risk - Fraud: -0.1632815787296096


In [12]:
test_anomaly_risk = -test_anomaly_score

In [14]:
fraud_risk = test_anomaly_risk[y_test_anomaly == 1]
legit_risk = test_anomaly_risk[y_test_anomaly == 0]

print("Legitimate transactions:", len(legit_risk))
print("Fraud transactions:", len(fraud_risk))

print("\nAverage anomaly risk:")
print("Legitimate:", np.mean(legit_risk))
print("Fraud:", np.mean(fraud_risk))

print("\nMedian anomaly risk:")
print("Legitimate:", np.median(legit_risk))
print("Fraud:", np.median(fraud_risk))

Legitimate transactions: 88214
Fraud transactions: 1252

Average anomaly risk:
Legitimate: -0.23997336734264765
Fraud: -0.1632815787296096

Median anomaly risk:
Legitimate: -0.25575006252696875
Fraud: -0.17155062525756648


In [15]:
from sklearn.metrics import roc_auc_score, average_precision_score

isolation_roc_auc = roc_auc_score(
    y_test_anomaly,
    test_anomaly_risk
)

isolation_pr_auc = average_precision_score(
    y_test_anomaly,
    test_anomaly_risk
)

print("Isolation Forest ROC-AUC:", isolation_roc_auc)
print("Isolation Forest PR-AUC:", isolation_pr_auc)

Isolation Forest ROC-AUC: 0.8519359298774669
Isolation Forest PR-AUC: 0.0895754758255395


In [16]:
import os
import joblib

os.makedirs("../models_saved", exist_ok=True)

ISOLATION_FILE = "../models_saved/isolation_forest.joblib"

joblib.dump(
    isolation_model,
    ISOLATION_FILE
)

print("Isolation Forest saved successfully.")
print(ISOLATION_FILE)

Isolation Forest saved successfully.
../models_saved/isolation_forest.joblib


In [17]:
import pandas as pd
import numpy as np

VALIDATION_FILE = "../data/processed/validation_data.csv"

validation_df = pd.read_csv(VALIDATION_FILE)

print("Validation shape:", validation_df.shape)

Validation shape: (191147, 25)


In [18]:
validation_df = pd.get_dummies(
    validation_df,
    columns=["type"],
    dtype=int
)

In [19]:
validation_df = validation_df.reindex(
    columns=train_df.columns,
    fill_value=0
)

In [20]:
X_validation_anomaly = (
    validation_df[ANOMALY_FEATURES]
    .fillna(0)
)

y_validation_anomaly = validation_df["isFraud"]

print("Validation anomaly shape:", X_validation_anomaly.shape)

Validation anomaly shape: (191147, 26)


In [21]:
validation_anomaly_score = (
    isolation_model.decision_function(
        X_validation_anomaly
    )
)

validation_anomaly_risk = -validation_anomaly_score

print(
    "Validation anomaly risk range:",
    validation_anomaly_risk.min(),
    "to",
    validation_anomaly_risk.max()
)

Validation anomaly risk range: -0.29150094031253565 to 0.06295718530316796


In [25]:
from xgboost import XGBClassifier

MODEL_FILE = "../models_saved/behaviour_aware_xgboost.json"

behaviour_model = XGBClassifier()

behaviour_model.load_model(MODEL_FILE)

print("Behaviour-Aware XGBoost loaded successfully.")

Behaviour-Aware XGBoost loaded successfully.


In [26]:
X_validation_anomaly = validation_df[ANOMALY_FEATURES].fillna(0)

y_validation_anomaly = validation_df["isFraud"]

print("Validation features:", X_validation_anomaly.shape)

Validation features: (191147, 26)


In [27]:
validation_xgb_probability = (
    behaviour_model.predict_proba(
        X_validation_anomaly
    )[:, 1]
)

print(
    "XGBoost probability range:",
    validation_xgb_probability.min(),
    "to",
    validation_xgb_probability.max()
)

XGBoost probability range: 3.4187937e-12 to 0.9999999


In [28]:
from sklearn.preprocessing import MinMaxScaler

anomaly_scaler = MinMaxScaler()

validation_anomaly_normalized = (
    anomaly_scaler.fit_transform(
        validation_anomaly_risk.reshape(-1, 1)
    ).ravel()
)

print(
    "Normalized anomaly range:",
    validation_anomaly_normalized.min(),
    "to",
    validation_anomaly_normalized.max()
)

Normalized anomaly range: 0.0 to 1.0


In [29]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

weights = [
    (0.9, 0.1),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5)
]

fusion_results = []

for xgb_weight, anomaly_weight in weights:

    fusion_score = (
        xgb_weight * validation_xgb_probability
        +
        anomaly_weight * validation_anomaly_normalized
    )

    pr_auc = average_precision_score(
        y_validation_anomaly,
        fusion_score
    )

    roc_auc = roc_auc_score(
        y_validation_anomaly,
        fusion_score
    )

    fusion_results.append({
        "xgb_weight": xgb_weight,
        "anomaly_weight": anomaly_weight,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc
    })

fusion_results_df = pd.DataFrame(fusion_results)

print(fusion_results_df)

   xgb_weight  anomaly_weight    PR_AUC   ROC_AUC
0         0.9             0.1  0.989574  0.999852
1         0.8             0.2  0.985197  0.999580
2         0.7             0.3  0.981861  0.999219
3         0.6             0.4  0.978570  0.998787
4         0.5             0.5  0.974441  0.998261


In [30]:
BEST_XGB_WEIGHT = 0.9
BEST_ANOMALY_WEIGHT = 0.1

validation_fusion_score = (
    BEST_XGB_WEIGHT * validation_xgb_probability
    +
    BEST_ANOMALY_WEIGHT * validation_anomaly_normalized
)

print("Best XGBoost weight:", BEST_XGB_WEIGHT)
print("Best Isolation Forest weight:", BEST_ANOMALY_WEIGHT)

print(
    "Fusion PR-AUC:",
    average_precision_score(
        y_validation_anomaly,
        validation_fusion_score
    )
)

print(
    "Fusion ROC-AUC:",
    roc_auc_score(
        y_validation_anomaly,
        validation_fusion_score
    )
)

Best XGBoost weight: 0.9
Best Isolation Forest weight: 0.1
Fusion PR-AUC: 0.989573612471066
Fusion ROC-AUC: 0.999851740529778


In [31]:
from sklearn.metrics import precision_score, recall_score, f1_score

fusion_thresholds = [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5
]

for threshold in fusion_thresholds:

    fusion_predictions = (
        validation_fusion_score >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation_anomaly,
        fusion_predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation_anomaly,
        fusion_predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation_anomaly,
        fusion_predictions,
        zero_division=0
    )

    print(
        f"Threshold={threshold:.1f} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | "
        f"F1={f1:.4f}"
    )

Threshold=0.1 | Precision=0.9101 | Recall=0.9949 | F1=0.9506
Threshold=0.2 | Precision=0.9125 | Recall=0.9898 | F1=0.9496
Threshold=0.3 | Precision=0.9150 | Recall=0.9856 | F1=0.9490
Threshold=0.4 | Precision=0.9205 | Recall=0.9814 | F1=0.9500
Threshold=0.5 | Precision=0.9219 | Recall=0.9805 | F1=0.9503


In [32]:
from sklearn.metrics import precision_score, recall_score, f1_score

fusion_thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

for threshold in fusion_thresholds:

    fusion_predictions = (
        validation_fusion_score >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation_anomaly,
        fusion_predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation_anomaly,
        fusion_predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation_anomaly,
        fusion_predictions,
        zero_division=0
    )

    print(
        f"Threshold={threshold:.1f} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | "
        f"F1={f1:.4f}"
    )

Threshold=0.1 | Precision=0.9101 | Recall=0.9949 | F1=0.9506
Threshold=0.2 | Precision=0.9125 | Recall=0.9898 | F1=0.9496
Threshold=0.3 | Precision=0.9150 | Recall=0.9856 | F1=0.9490
Threshold=0.4 | Precision=0.9205 | Recall=0.9814 | F1=0.9500
Threshold=0.5 | Precision=0.9219 | Recall=0.9805 | F1=0.9503


In [33]:
FINAL_FUSION_THRESHOLD = 0.5

print(
    "Selected fusion threshold:",
    FINAL_FUSION_THRESHOLD
)

Selected fusion threshold: 0.5


In [34]:
BEST_XGB_WEIGHT = 0.9
BEST_ANOMALY_WEIGHT = 0.1

FINAL_FUSION_THRESHOLD = 0.1

In [35]:
test_xgb_probability = (
    behaviour_model.predict_proba(
        X_test_anomaly
    )[:, 1]
)

In [36]:
test_anomaly_score = isolation_model.decision_function(
    X_test_anomaly
)

test_anomaly_risk = -test_anomaly_score

In [37]:
test_anomaly_normalized = (
    anomaly_scaler.transform(
        test_anomaly_risk.reshape(-1, 1)
    ).ravel()
)

In [38]:
test_fusion_score = (
    BEST_XGB_WEIGHT * test_xgb_probability
    +
    BEST_ANOMALY_WEIGHT * test_anomaly_normalized
)

print(
    "Test fusion score range:",
    test_fusion_score.min(),
    "to",
    test_fusion_score.max()
)

Test fusion score range: 0.0009337577853738566 to 0.9843780317550477


In [41]:
BEST_XGB_WEIGHT = 0.9
BEST_ANOMALY_WEIGHT = 0.1

FINAL_FUSION_THRESHOLD = 0.1

print("XGBoost weight:", BEST_XGB_WEIGHT)
print("Isolation Forest weight:", BEST_ANOMALY_WEIGHT)
print("Final threshold:", FINAL_FUSION_THRESHOLD)

XGBoost weight: 0.9
Isolation Forest weight: 0.1
Final threshold: 0.1


In [43]:
from xgboost import XGBClassifier

MODEL_FILE = "../models_saved/behaviour_aware_xgboost.json"

behaviour_model = XGBClassifier()
behaviour_model.load_model(MODEL_FILE)

print("Behaviour-Aware XGBoost loaded successfully.")

Behaviour-Aware XGBoost loaded successfully.


In [44]:
X_test_xgb = test_df[ANOMALY_FEATURES].fillna(0)

print("X_test shape:", X_test_xgb.shape)

X_test shape: (89466, 26)


In [45]:
test_xgb_probability = behaviour_model.predict_proba(
    X_test_xgb
)[:, 1]

print(
    "Test XGBoost probability shape:",
    test_xgb_probability.shape
)

Test XGBoost probability shape: (89466,)


In [46]:
test_anomaly_score = isolation_model.decision_function(
    X_test_anomaly
)

test_anomaly_risk = -test_anomaly_score

In [47]:
test_anomaly_normalized = (
    anomaly_scaler.transform(
        test_anomaly_risk.reshape(-1, 1)
    ).ravel()
)

In [48]:
BEST_XGB_WEIGHT = 0.9
BEST_ANOMALY_WEIGHT = 0.1
FINAL_FUSION_THRESHOLD = 0.1

test_fusion_score = (
    BEST_XGB_WEIGHT * test_xgb_probability
    +
    BEST_ANOMALY_WEIGHT * test_anomaly_normalized
)

test_fusion_predictions = (
    test_fusion_score >= FINAL_FUSION_THRESHOLD
).astype(int)

print("Fusion predictions created.")
print("Predicted fraud:", (test_fusion_predictions == 1).sum())
print("Predicted legitimate:", (test_fusion_predictions == 0).sum())

Fusion predictions created.
Predicted fraud: 1312
Predicted legitimate: 88154


In [49]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

print(
    classification_report(
        y_test_anomaly,
        test_fusion_predictions,
        target_names=["Legitimate", "Fraud"],
        digits=4
    )
)

print(
    "Fusion Test PR-AUC:",
    average_precision_score(
        y_test_anomaly,
        test_fusion_score
    )
)

print(
    "Fusion Test ROC-AUC:",
    roc_auc_score(
        y_test_anomaly,
        test_fusion_score
    )
)

print("\nFusion Confusion Matrix:")
print(
    confusion_matrix(
        y_test_anomaly,
        test_fusion_predictions
    )
)

              precision    recall  f1-score   support

  Legitimate     1.0000    0.9993    0.9997     88214
       Fraud     0.9543    1.0000    0.9766      1252

    accuracy                         0.9993     89466
   macro avg     0.9771    0.9997    0.9881     89466
weighted avg     0.9994    0.9993    0.9993     89466

Fusion Test PR-AUC: 0.9969290136423676
Fusion Test ROC-AUC: 0.9999612382493314

Fusion Confusion Matrix:
[[88154    60]
 [    0  1252]]


In [50]:
print("XGBoost probability:", test_xgb_probability.shape)
print("Anomaly risk:", test_anomaly_risk.shape)
print("Normalized anomaly:", test_anomaly_normalized.shape)
print("Fusion score:", test_fusion_score.shape)
print("Fusion predictions:", test_fusion_predictions.shape)
print("Test labels:", y_test_anomaly.shape)

XGBoost probability: (89466,)
Anomaly risk: (89466,)
Normalized anomaly: (89466,)
Fusion score: (89466,)
Fusion predictions: (89466,)
Test labels: (89466,)


In [51]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

print("========== FUSION TEST RESULTS ==========\n")

print(
    classification_report(
        y_test_anomaly,
        test_fusion_predictions,
        target_names=["Legitimate", "Fraud"],
        digits=4
    )
)

fusion_test_pr_auc = average_precision_score(
    y_test_anomaly,
    test_fusion_score
)

fusion_test_roc_auc = roc_auc_score(
    y_test_anomaly,
    test_fusion_score
)

print("Fusion Test PR-AUC:", fusion_test_pr_auc)
print("Fusion Test ROC-AUC:", fusion_test_roc_auc)

cm = confusion_matrix(
    y_test_anomaly,
    test_fusion_predictions
)

print("\nFusion Confusion Matrix:")
print(cm)

========== FUSION TEST RESULTS ==========

              precision    recall  f1-score   support

  Legitimate     1.0000    0.9993    0.9997     88214
       Fraud     0.9543    1.0000    0.9766      1252

    accuracy                         0.9993     89466
   macro avg     0.9771    0.9997    0.9881     89466
weighted avg     0.9994    0.9993    0.9993     89466

Fusion Test PR-AUC: 0.9969290136423676
Fusion Test ROC-AUC: 0.9999612382493314

Fusion Confusion Matrix:
[[88154    60]
 [    0  1252]]


In [52]:
import json
import os

fusion_config = {
    "xgb_weight": 0.9,
    "isolation_forest_weight": 0.1,
    "threshold": 0.1
}

CONFIG_FILE = "../models_saved/fusion_config.json"

with open(CONFIG_FILE, "w") as f:
    json.dump(fusion_config, f, indent=4)

print("Fusion configuration saved:")
print(CONFIG_FILE)

Fusion configuration saved:
../models_saved/fusion_config.json


In [53]:
import json
import os

final_model_config = {
    "primary_model": "Behaviour-Aware XGBoost",
    "xgboost_model_file": "behaviour_aware_xgboost.json",

    "xgboost_features": 26,

    "threshold": 0.5,

    "validation_pr_auc": 0.9956991314256074,
    "validation_roc_auc": 0.9999737688606576,

    "test_pr_auc": 0.9979555153653352,
    "test_roc_auc": 0.999973185488296,

    "test_fraud_precision": 0.9594,
    "test_fraud_recall": 1.0000,
    "test_fraud_f1": 0.9793,

    "isolation_forest": {
        "enabled_for_anomaly_monitoring": True,
        "roc_auc": 0.8519359298774669,
        "pr_auc": 0.0895754758255395
    },

    "gru": {
        "experimental": True,
        "used_in_final_prediction": False,
        "roc_auc": 0.5000850229561982,
        "pr_auc": 0.0002700459078043267
    }
}

CONFIG_FILE = "../models_saved/final_model_config.json"

with open(CONFIG_FILE, "w") as f:
    json.dump(
        final_model_config,
        f,
        indent=4
    )

print("Final model configuration saved:")
print(CONFIG_FILE)

Final model configuration saved:
../models_saved/final_model_config.json
